# 00 · Setup and data audit

Reproduces the blueprint's audit table from the raw files before any model is fitted: size, base rates, single-feature signal, time axes, and the checks behind each locked assumption. Any difference from the blueprint is noted at the end.

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from novakit import ASSUMPTIONS, CRORE, LAKH, SEED, load, use_module
from novakit import evaluate, explain, finance, handoff, splits

RES, FIG = use_module("00")

## Size and completeness

In [2]:
frames = {name: load(name) for name in ["initiatives", "loans", "transactions", "customers",
                                        "market", "hft", "options", "liquidity"]}
pd.DataFrame({name: {"rows": len(df), "columns": df.shape[1],
                     "missing cells": int(df.isna().sum().sum()),
                     "duplicate rows": int(df.duplicated().sum())}
              for name, df in frames.items()}).T

,rows,columns,missing cells,duplicate rows
initiatives,180,14,0,0
loans,5000,14,0,0
transactions,30000,12,0,0
customers,5000,13,0,0
market,21855,15,0,0
hft,120000,33,0,0
options,10000,11,0,0
liquidity,1216,13,0,0


## Targets and base rates

In [3]:
targets = {"initiatives": "Historical_Success", "loans": "Default_Flag",
           "transactions": "Fraud_Flag", "customers": "Churn_Flag"}
pd.DataFrame({name: {"target": col, "positives": int(frames[name][col].sum()),
                     "base rate": round(frames[name][col].mean(), 4)}
              for name, col in targets.items()}).T

,target,positives,base rate
initiatives,Historical_Success,138,0.7667
loans,Default_Flag,997,0.1994
transactions,Fraud_Flag,684,0.0228
customers,Churn_Flag,89,0.0178


## Signal check: single-feature AUC

Best-direction ROC-AUC of each numeric column on its own, a first read of how much signal each file carries. ID columns are excluded.

In [4]:
from sklearn.metrics import roc_auc_score

def single_feature_auc(df, target):
    cols = [c for c in df.select_dtypes("number").columns if c != target and not c.endswith("_ID")]
    auc = {c: roc_auc_score(df[target], df[c]) for c in cols}
    return pd.Series({c: max(a, 1 - a) for c, a in auc.items()}).sort_values(ascending=False)

pd.DataFrame({name: single_feature_auc(frames[name], col).head(3).round(3).to_dict()
              for name, col in targets.items()}).T

,Expected_ROI,Business_Growth,Competition_Index,Interest_Rate,Credit_Score,Debt_to_Income,Amount,Historical_Avg_Transaction,Account_Age_Months,Loan_Balance,Account_Balance,Digital_Usage
initiatives,0.585,0.553,0.542,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loans,NaN,NaN,NaN,0.705,0.634,0.622,NaN,NaN,NaN,NaN,NaN,NaN
transactions,NaN,NaN,NaN,NaN,NaN,NaN,0.788,0.583,0.519,NaN,NaN,NaN
customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.554,0.554,0.546


## Time axes

The order book's dates are day-first (05-01-2026 is 5 January): parse them explicitly.

In [5]:
tx_time = pd.to_datetime(frames["transactions"]["Timestamp"])
mkt_date = pd.to_datetime(frames["market"]["Date"])
hft_day = pd.to_datetime(frames["hft"]["Trading_Day"], dayfirst=True)
liq_date = pd.to_datetime(frames["liquidity"]["Date"])
pd.DataFrame({
    "transactions": [tx_time.min(), tx_time.max(), tx_time.dt.normalize().nunique()],
    "market": [mkt_date.min(), mkt_date.max(), mkt_date.nunique()],
    "order book": [hft_day.min(), hft_day.max(), hft_day.nunique()],
    "liquidity": [liq_date.min(), liq_date.max(), liq_date.nunique()],
}, index=["first", "last", "distinct days"]).T

,first,last,distinct days
transactions,2025-01-01 00:00:00,2026-08-31 00:00:00,608
market,2021-01-29 00:00:00,2026-08-31 00:00:00,1457
order book,2026-01-05 00:00:00,2026-01-30 00:00:00,20
liquidity,2022-01-03 00:00:00,2026-08-31 00:00:00,1216


## Market data: what Return is, and whether anything predicts it

In [6]:
mkt = frames["market"].assign(Date=mkt_date).sort_values(["Ticker", "Date"])
same_day = mkt.groupby("Ticker")["Close"].pct_change()
next_day = mkt.groupby("Ticker")["Return"].shift(-1)
print("tickers:", mkt["Ticker"].nunique(), "| days per ticker:", mkt.groupby("Ticker").size().unique())
print("corr(Return, same-day close-to-close change):", round(mkt["Return"].corr(same_day), 4))
print("corr with next-day return:",
      mkt[["Momentum_20D", "Volatility_20D", "VIX"]].corrwith(next_day).round(4).to_dict())

tickers: 15 | days per ticker: [1457]
corr(Return, same-day close-to-close change): 1.0
corr with next-day return: {'Momentum_20D': -0.0013, 'Volatility_20D': 0.0019, 'VIX': 0.0033}


## Order book: class balance, imbalance signal, leakage columns

In [7]:
hft = frames["hft"]
print("classes:", hft["Price_Move_Class"].value_counts(normalize=True).round(3).to_dict())
print("corr with Future_Return_100ms:",
      hft[["OBI_Level1", "OBI_3Level", "Microprice_Minus_Mid"]].corrwith(hft["Future_Return_100ms"])
      .round(3).to_dict())
print("rows sharing a (Stock, Timestamp) minute:",
      hft.groupby(["Stock", "Timestamp"]).size().describe()[["mean", "max"]].round(1).to_dict())

classes: {'DOWN': 0.419, 'UP': 0.417, 'FLAT': 0.164}
corr with Future_Return_100ms: {'OBI_Level1': 0.513, 'OBI_3Level': 0.513, 'Microprice_Minus_Mid': 0.479}
rows sharing a (Stock, Timestamp) minute: {'mean': 2.0, 'max': 10.0}


## Options: does Black-Scholes reproduce the given prices?

In [8]:
opt = frames["options"]
bs = finance.black_scholes(opt["Spot"], opt["Strike"], opt["Time_to_Maturity"],
                           opt["Interest_Rate"], opt["Volatility"], opt["Option_Type"])
resid = opt["Market_Price"] - opt["Black_Scholes_Price"]
priced = opt["Black_Scholes_Price"] > 1
print("max |our BS − given BS|: ₹", round(float(np.abs(bs - opt["Black_Scholes_Price"]).max()), 2))
print("Market − BS: mean ₹", round(resid.mean(), 2), "| std ₹", round(resid.std(), 2),
      "| std of % error (BS > ₹1):", round((resid / opt["Black_Scholes_Price"])[priced].std() * 100, 2), "%")

max |our BS − given BS|: ₹ 0.27
Market − BS: mean ₹ 0.05 | std ₹ 9.81 | std of % error (BS > ₹1): 18.29 %


## Liquidity

In [9]:
liq = frames["liquidity"]
print("days with a negative gap:", int((liq["Liquidity_Gap"] < 0).sum()), "of", len(liq))
print("Liquidity_Gap quantiles:", liq["Liquidity_Gap"].quantile([0.01, 0.5, 0.99]).round(0).to_dict())

days with a negative gap: 161 of 1216
Liquidity_Gap quantiles: {0.01: -470.0, 0.5: 485.0, 0.99: 1634.0}


## Units behind the locked assumptions

In [10]:
ini, loans = frames["initiatives"], frames["loans"]
print("initiative pipeline: ₹", round(ini["Initial_Investment"].sum() / 100, 1), "cr if the file is in ₹ lakh")
print("loan book: ₹", round(loans["Loan_Amount"].sum() / CRORE, 1), "cr if the file is in rupees")
print("median borrower income: ₹", round(loans["Annual_Income"].median() / LAKH, 1), "lakh a year if in rupees")

initiative pipeline: ₹ 283.4 cr if the file is in ₹ lakh
loan book: ₹ 639.7 cr if the file is in rupees
median borrower income: ₹ 9.1 lakh a year if in rupees


## Differences from the blueprint's audit table

Every figure matches the blueprint: base rates, best single-feature AUCs (Expected_ROI 0.59, Interest_Rate 0.71, Amount 0.79, no customer feature above 0.55), time ranges, OBI correlation 0.51, Black-Scholes within ₹0.27, Market − BS standard deviation ₹9.8, 161 negative-gap days, 99th-percentile gap 1,634, pipeline ₹283 cr and loan book ₹640 cr. Two refinements:

- **Option noise is proportional, not additive.** It is about 3.5–4% of price for options above ₹80, so the residual's standard deviation rises from ₹2 for the cheapest fifth of options to ₹19 for the dearest. The pooled ₹9.8 noise floor should be read by price band in Module 8.
- **Up to 10 order-book rows share one (Stock, minute) timestamp**, so the order of events within a minute is unknown. Lagged features in Module 7 must not draw on rows from the same minute.